In [0]:
# ── this is NOT a physical table ─────────────────────────────────────────────
# it is a VIEW — no data stored, computed on the fly when queried
# every Gold aggregation builds on top of this view

spark.sql("""
    CREATE OR REPLACE VIEW spotify_catalog.view.vw_base_fact AS
select f.stream_id,
u.user_sk,
t.track_sk,
a.artist_sk,
f.user_id,
t.track_id,
a.artist_id,
f.listen_duration,
t.duration_sec,
case when f.listen_duration is not null  and f.listen_duration>=0.2*t.duration_sec then 1
     else 0 end as is_meaningful_play,
case when f.listen_duration is null or f.listen_duration=0 then "Abandoned"
      when f.listen_duration<(t.duration_sec*0.2) then "Skipped"
      when f.listen_duration<(t.duration_sec*0.5) then "partial"
      else "Completed" end as play_category,
u.user_name,
u.country as user_country,
u.subscription_type,
f.device_type,
t.track_name,
t.album_name,
a.artist_name,
a.genre,
a.country as artist_country,
d.date_key,
d.date,
d.day,
d.month,
d.year,
d.weekday 
    FROM spotify_catalog.silver.FactStream     f
    LEFT JOIN spotify_catalog.view.v_DimUser   u  ON f.user_id    = u.user_id
    LEFT JOIN spotify_catalog.silver.DimTrack  t  ON f.track_id  = t.track_id
    LEFT JOIN spotify_catalog.silver.DimArtist a  ON t.artist_id = a.artist_id
    LEFT JOIN spotify_catalog.silver.DimDate   d  ON f.date_key  = d.date_key
""")

print("Base view vw_base_fact created")

In [0]:
%sql
select f.stream_id,
u.user_sk,
t.track_sk,
a.artist_sk,
f.user_id,
t.track_id,
a.artist_id,
f.listen_duration,
t.duration_sec,
case when f.listen_duration is not null  and f.listen_duration>=0.2*t.duration_sec then 1
     else 0 end as is_meaningful_play,
case when f.listen_duration is null or f.listen_duration=0 then "Abandoned"
      when f.listen_duration<(t.duration_sec*0.2) then "Skipped"
      when f.listen_duration<(t.duration_sec*0.5) then "partial"
      else "Completed" end as play_category,
u.user_name,
u.country as user_country,
u.subscription_type,
t.track_name,
t.album_name,
a.artist_name,
a.genre,
a.country as artist_country,
d.date_key,
d.date,
d.day,
d.month,
d.year,
d.weekday 
    FROM spotify_catalog.silver.FactStream     f
    LEFT JOIN spotify_catalog.view.v_DimUser   u  ON f.user_id    = u.user_id
    LEFT JOIN spotify_catalog.silver.DimTrack  t  ON f.track_id  = t.track_id
    LEFT JOIN spotify_catalog.silver.DimArtist a  ON t.artist_id = a.artist_id
    LEFT JOIN spotify_catalog.silver.DimDate   d  ON f.date_key  = d.date_key

In [0]:
%sql
select * FROM spotify_catalog.silver.FactStream     f
    LEFT JOIN spotify_catalog.view.v_DimUser   u  ON f.user_id    = u.user_id
    LEFT JOIN spotify_catalog.silver.DimTrack  t  ON f.track_id  = t.track_id
    LEFT JOIN spotify_catalog.silver.DimArtist a  ON t.artist_id = a.artist_id
    LEFT JOIN spotify_catalog.silver.DimDate   d  ON f.date_key  = d.date_key